# RAG end-to-end on Cloud SQL — embeddings x chunking x storage, benchmarked (psycopg2 edition)

**New to this? Start here.** RAG = *Retrieval-Augmented Generation*. To answer a question about your documents you: **(1)** cut documents into pieces (*chunking*), **(2)** turn each piece into a vector of numbers that captures its meaning (*embedding*), **(3)** store those vectors in Postgres tables (*pgvector*, optionally with a search *index*), **(4)** for a question, find the closest pieces (*vector search*), **(5)** let Gemini write the answer from those pieces.

The quality depends on **which embedding model**, **which chunking method**, and **how you store/index the vectors** you pick — and there's no universal best, it depends on *your* data. So this notebook tries several of each, **measures them**, and tells you the winner.

Same connection style as your `vector_search_psycopg2` notebook (`DB_CONFIG` + `psycopg2` + `google-genai` Vertex). Run cells top-to-bottom.

| Step | What happens |
|---|---|
| 1-3 | Install, configure, **connect to Cloud SQL** |
| 4 | Enable pgvector |
| 5 | Vertex embedding function (works for any model) |
| 6 | **Chunking** — three techniques, side by side |
| 7 | Your documents + a small *gold set* (for scoring) |
| 8 | **Ingest** — chunk + embed + store (in a table, with a chosen index type), once per combo |
| 9 | Vector search |
| 10 | **Benchmark** — recall@k, MRR & search speed leaderboard across model x chunking x storage → the best combo |
| 11 | Answer questions with Gemini using the winner |
| 12 | Cleanup + troubleshooting |

## Step 1 — Install

In [1]:
# Install everything this notebook needs:
#   google-genai      -> talk to Vertex AI (embeddings + Gemini chat)
#   psycopg2-binary    -> connect to PostgreSQL / Cloud SQL
#   tiktoken           -> token-based chunking in Step 6
#   pandas             -> not strictly required, kept for optional exploration
%pip install -q google-genai psycopg2-binary tiktoken pandas
print("Installed. If pip shows a 'restart kernel' notice, restart the kernel now.")

Note: you may need to restart the kernel to use updated packages.
Installed. If pip shows a 'restart kernel' notice, restart the kernel now.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Configuration

**Copy `DB_CONFIG` from your working `vector_search_psycopg2` / `sql_query_expert` notebook.** If you connect through the Cloud SQL Auth Proxy, keep `host="localhost"` and make sure the proxy is running.

`EMBEDDERS`, `CHUNKERS`, and `INDEXERS` are the three methods we compare — add or remove freely.

In [ ]:
# ---------- Google Cloud / Vertex AI ----------
PROJECT_ID = "div-aais-rfpiq-usc1-uat"     # CHANGE - your GCP project id
LOCATION   = "us-central1"                 # Vertex AI region
CHAT_MODEL = "gemini-2.5-flash"            # model used to write the final answer in Step 11

# ---------- PostgreSQL / Cloud SQL (copy from your other notebook) ----------
DB_CONFIG = {
    "host":     "10.151.179.4",           # CHANGE if needed - Cloud SQL private IP, or "localhost" if using the Auth Proxy
    "port":     5432,
    "dbname":   "postgres",       # CHANGE - target database name
    "user":     "postgres",            # CHANGE - db user
    "password": "Rfpiq-usc1",       # CHANGE - db password
}

# ---------- Methods to compare ----------
# 1) Embedding models (all Vertex / google-genai). Each model turns a piece of text into
#    a vector of numbers ("dimensions") that captures its meaning. The vector's dimension
#    is auto-detected in Step 8, so you don't need to know it up front - just list names.
EMBEDDERS = [
    "text-embedding-004",     # 768-dim, general purpose
    "text-embedding-005",     # 768-dim, newer general purpose
    "gemini-embedding-001",   # 3072-dim, highest quality but larger/slower to store & search
]

# 2) Chunking methods - the functions defined in Step 6; referenced here by name.
CHUNKERS = ["fixed_400", "sentence", "token_256"]

# 3) Storage / index methods used when we save the vectors in Step 8 (the "storing"
#    dimension). An index is how Postgres organizes vectors internally so a search
#    doesn't have to compare against every single row ("brute force"). Options here
#    trade off search speed vs. exactness vs. dimension limits:
#      - "hnsw"     -> approximate nearest-neighbour graph; usually the best default,
#                       fastest at scale, most commonly used in production
#      - "ivfflat"  -> approximate, cheaper to build but needs a "lists" tuning
#                       parameter and a data-dependent training step (see Step 8)
#      - "none"     -> no index at all = exact brute-force scan. Slower as data grows,
#                       but always 100% correct AND has no vector-dimension ceiling
#                       (hnsw/ivfflat top out at 2000 dims - see Step 10 notes)
INDEXERS = ["hnsw", "ivfflat", "none"]

TOP_K = 3   # retrieve/score the top-3 chunks per question
print("Config loaded.")

Config loaded.


## Step 3 — Connect to Cloud SQL (same pattern as your notebook)

In [3]:
import psycopg2
import psycopg2.extras

def get_conn():
    # Opens a fresh connection using the DB_CONFIG dict above. Called per-operation
    # (rather than reusing one global connection) to keep the notebook's cells independent.
    return psycopg2.connect(**DB_CONFIG)

try:
    # `with get_conn() as conn` auto-commits/rolls back and closes the connection;
    # `with conn.cursor() as cur` auto-closes the cursor - both clean up even on error.
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute("SELECT current_database(), version();")
        dbname, version = cur.fetchone()
    print(f"Connected to '{dbname}'")
    print("Server:", version.split(',')[0])
except Exception as e:
    # Fail loudly with a hint instead of letting a cryptic psycopg2 error stop the notebook.
    raise SystemExit(f"Could not connect: {e}\n  -> Check DB_CONFIG. If host is 'localhost', is the Cloud SQL Auth Proxy running?")

Connected to 'postgres'
Server: PostgreSQL 18.4 on x86_64-pc-linux-gnu


## Step 4 — Enable pgvector

One-time. Cloud SQL for PostgreSQL supports the `vector` extension.

In [4]:
with get_conn() as conn, conn.cursor() as cur:
    # IF NOT EXISTS makes this safe to re-run on every notebook execution.
    # This adds the `vector` column type and the <=> / <-> / <#> distance operators
    # used later, plus the hnsw/ivfflat index access methods used in Step 8.
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    conn.commit()
print("pgvector ready.")

pgvector ready.


## Step 5 — Vertex embedding function

**The golden rule:** documents and questions must be embedded by the *same* model. We pass `task_type` so the model knows whether it's embedding a *document* (`RETRIEVAL_DOCUMENT`) or a *search query* (`RETRIEVAL_QUERY`) — this improves results. The function batches and falls back to one-at-a-time if a model rejects batches.

In [5]:
from google import genai
from google.genai.types import EmbedContentConfig

# One shared Vertex client for the whole notebook (embeddings in this step, chat in Step 11).
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

def embed_texts(model: str, texts: list[str], task_type: str) -> list[list[float]]:
    """Embed a list of strings with `model`. task_type tells the model whether these
    are documents to be searched (RETRIEVAL_DOCUMENT) or a question doing the searching
    (RETRIEVAL_QUERY) - using the right one improves match quality."""
    out: list[list[float]] = []
    for i in range(0, len(texts), 50):          # send at most 50 texts per API call
        batch = texts[i:i + 50]
        cfg = EmbedContentConfig(task_type=task_type)
        try:
            resp = genai_client.models.embed_content(model=model, contents=batch, config=cfg)
            out.extend([e.values for e in resp.embeddings])
        except Exception:
            # Some models/quotas only accept one input per call - retry the batch
            # one text at a time instead of failing the whole thing.
            for t in batch:
                resp = genai_client.models.embed_content(model=model, contents=[t], config=cfg)
                out.append(resp.embeddings[0].values)
    return out

# Smoke test: embed one word and report the vector size (its "dimension").
_probe = embed_texts("text-embedding-004", ["hello"], "RETRIEVAL_QUERY")[0]
print(f"Vertex works — text-embedding-004 returns {len(_probe)} dims.")

Vertex works — text-embedding-004 returns 768 dims.


## Step 6 — Chunking methods (three techniques)

Each function turns one document into a list of text pieces:

- **`fixed_400`** — a sliding window of ~400 characters with overlap. Simplest; predictable size.
- **`sentence`** — packs whole sentences together up to a size. Respects natural boundaries → cleaner meaning.
- **`token_256`** — windows measured in *model tokens* (not characters). Best for staying within model limits.

Overlap means neighbouring pieces share some text, so an answer that straddles a boundary isn't lost.

In [6]:
import re
import tiktoken

# cl100k_base is the tokenizer GPT-3.5/4 use. We reuse it purely as a consistent way to
# count/split "tokens" for chunk_tokens() below - it doesn't need to match the embedding model.
_enc = tiktoken.get_encoding("cl100k_base")

# Heuristic sentence-boundary regex: a period/!/? followed by whitespace and then a
# capital letter or digit. Not perfect (abbreviations can fool it) but good enough here.
_SENT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9])")

def chunk_fixed(text, size=400, overlap=60):
    """Sliding window over raw characters. Simple and predictable, but can cut a
    chunk off mid-word or mid-sentence."""
    text = text.strip()
    step = max(1, size - overlap)               # how far the window slides each time
    # Slice the text every `step` characters into `size`-character pieces; drop any
    # slice that's empty (can happen at the very end of the text).
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]

def chunk_sentence(text, size=500):
    """Pack whole sentences together until adding the next one would exceed `size`
    characters, then start a new chunk. Keeps chunk boundaries on sentence edges."""
    sents = [s.strip() for s in _SENT.split(text.strip()) if s.strip()]
    chunks, buf = [], ""
    for s in sents:
        if buf and len(buf) + len(s) > size:
            chunks.append(buf); buf = s          # current chunk is full - flush it, start a new one
        else:
            buf = f"{buf} {s}".strip()           # still room - keep appending to the current chunk
    if buf:
        chunks.append(buf)                       # flush whatever's left over
    return chunks

def chunk_tokens(text, size=256, overlap=40):
    """Sliding window measured in model *tokens* instead of characters - useful when
    you need to guarantee a chunk fits under a model's context/token limit."""
    toks = _enc.encode(text)
    step = max(1, size - overlap)
    return [_enc.decode(toks[i:i + size]) for i in range(0, len(toks), step)]

# Lookup table so the rest of the notebook can pick a chunker by name (see CHUNKERS in Step 2).
CHUNKER_FUNCS = {
    "fixed_400": lambda t: chunk_fixed(t, 400, 60),
    "sentence":  lambda t: chunk_sentence(t, 500),
    "token_256": lambda t: chunk_tokens(t, 256, 40),
}

# Quick sanity check: show what each chunker does to one short example string.
demo = "Acme Foods quoted 3.20 USD per case for romaine. Payment terms were Net-30. Delivery was twice weekly."
for name, fn in CHUNKER_FUNCS.items():
    print(name, "->", fn(demo))

fixed_400 -> ['Acme Foods quoted 3.20 USD per case for romaine. Payment terms were Net-30. Delivery was twice weekly.']
sentence -> ['Acme Foods quoted 3.20 USD per case for romaine. Payment terms were Net-30. Delivery was twice weekly.']
token_256 -> ['Acme Foods quoted 3.20 USD per case for romaine. Payment terms were Net-30. Delivery was twice weekly.']


## Step 7 — Your documents + gold set, from SQL

Instead of hand-typing document text into Python, this step reads/writes real documents
in Postgres. There are two ways to get documents into `docs` — pick whichever fits you
with the `DOCS_SOURCE` switch in **7c**. You can leave both **7a** and **7b** in the
notebook (they don't conflict); only the one selected by `DOCS_SOURCE` actually feeds
the benchmark.

- **7a — Managed table** (recommended if you don't already have a documents table):
  creates a simple `rag_source_documents` table and writes your text into it. Call
  `write_source_docs({...})` again any time to add or update documents — no notebook
  code changes needed after that, it's just data.
- **7b — Point at a table you already have**: if bid/RFP text already lives in a table
  in this database, tell the notebook the table + column names and it reads straight
  from there.

Either way, this is *separate* from the `rag_bench_*` tables Step 8 creates — those hold
chunked text + vectors per (model, chunker, indexer) combo; this step's table holds your
raw, un-chunked source text.

`gold` (in **7c**) is unchanged in spirit: a `question → doc_id` map used to score which
model + chunking + storage combo is best. Its `doc_id` values must match whatever `docs`
ends up containing, whichever source you use — the cell checks this for you and warns if
any are missing.

In [7]:
# ---------------------------------------------------------------------------
# 7a. ENHANCED: Managed source-documents table with metadata columns
# ---------------------------------------------------------------------------
from psycopg2.extras import execute_values

SOURCE_TABLE = "rag_source_documents"

def ensure_source_table():
    """Create the table that holds raw document text + useful metadata.
    Separate from the per-combo rag_bench_* tables that hold chunks + vectors."""
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS {SOURCE_TABLE} (
                doc_id text PRIMARY KEY,
                title text,
                content text NOT NULL,
                supplier_name text,
                category text,
                bid_date date,
                document_type text,
                status text,
                delivery_terms text,
                price_range text,
                created_at timestamptz NOT NULL DEFAULT now(),
                updated_at timestamptz NOT NULL DEFAULT now()
            )
        """)
        conn.commit()

def write_source_docs(doc_map: dict):
    """Insert or update documents. Supports two formats:
    
    Format 1 - Simple (doc_id -> text):
        write_source_docs({
            "doc-id": "full document text here..."
        })
    
    Format 2 - Rich metadata (doc_id -> dict with 'content' key):
        write_source_docs({
            "doc-id": {
                "title": "...",
                "content": "...",
                "supplier_name": "...",
                "category": "...",
                "bid_date": "2024-05-01",
                "document_type": "quote",
                "status": "active",
                "delivery_terms": "twice weekly",
                "price_range": "$2-4 per unit"
            }
        })
    """
    ensure_source_table()
    rows = []
    
    for doc_id, doc_data in doc_map.items():
        # Handle both simple string and rich dict formats
        if isinstance(doc_data, str):
            # Simple format: just content
            title = doc_id
            content = doc_data
            supplier_name = None
            category = None
            bid_date = None
            document_type = None
            status = None
            delivery_terms = None
            price_range = None
        else:
            # Rich format: extract metadata
            title = doc_data.get("title", doc_id)
            content = doc_data.get("content", "")
            supplier_name = doc_data.get("supplier_name")
            category = doc_data.get("category")
            bid_date = doc_data.get("bid_date")
            document_type = doc_data.get("document_type")
            status = doc_data.get("status")
            delivery_terms = doc_data.get("delivery_terms")
            price_range = doc_data.get("price_range")
        
        rows.append((doc_id, title, content, supplier_name, category, bid_date, 
                    document_type, status, delivery_terms, price_range))
    
    with get_conn() as conn, conn.cursor() as cur:
        execute_values(
            cur,
            f"""INSERT INTO {SOURCE_TABLE} 
               (doc_id, title, content, supplier_name, category, bid_date, 
                document_type, status, delivery_terms, price_range) VALUES %s
               ON CONFLICT (doc_id) DO UPDATE
               SET content = EXCLUDED.content, 
                   supplier_name = EXCLUDED.supplier_name,
                   category = EXCLUDED.category,
                   bid_date = EXCLUDED.bid_date,
                   document_type = EXCLUDED.document_type,
                   status = EXCLUDED.status,
                   delivery_terms = EXCLUDED.delivery_terms,
                   price_range = EXCLUDED.price_range,
                   updated_at = now()""",
            rows,
        )
        conn.commit()
    print(f"Wrote {len(rows)} document(s) into {SOURCE_TABLE}.")

def load_source_docs() -> dict[str, str]:
    """Read every document back out of the managed table, as {doc_id: content}.
    (Only returns content for RAG - metadata stays in DB for filtering/discovery)"""
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(f"SELECT doc_id, content FROM {SOURCE_TABLE} ORDER BY doc_id")
        return {doc_id: content for doc_id, content in cur.fetchall()}

# ============================================================================
# SEED WITH REALISTIC SYSCO BID/RFP EXAMPLES (with metadata)
# ============================================================================
write_source_docs({
    "acme-produce-2024": {
        "title": "Acme Foods - Produce Quote",
        "content": (
            "Acme Foods bid on the produce category for Riverside USD on 2024-05-01. "
            "Romaine lettuce: 3.20 USD/case. Spinach: 4.10 USD/case. Carrots: 2.05 USD/case. "
            "Brussels sprouts: 2.85 USD/case. Bell peppers (mixed): 3.50 USD/case. "
            "Payment terms: Net-30. Delivery: twice weekly on Tuesdays and Thursdays. "
            "Minimum order: $100. Volume discount: 5% on orders over $500."
        ),
        "supplier_name": "Acme Foods Inc.",
        "category": "Produce",
        "bid_date": "2024-05-01",
        "document_type": "Supplier Quote",
        "status": "Active",
        "delivery_terms": "Twice weekly (Tue/Thu)",
        "price_range": "$2.05-$4.10 per case"
    },
    
    "globex-dairy-2024": {
        "title": "Globex Dairy - Dairy Products Quote",
        "content": (
            "Globex Dairy responded to the dairy RFP for Riverside USD on 2024-05-02. "
            "Whole milk (gallon): 1.85 USD/gallon. 2% milk (gallon): 1.75 USD/gallon. "
            "Skim milk (gallon): 1.65 USD/gallon. Cheddar block (lb): 3.40 USD/lb. "
            "Mozzarella (lb): 2.95 USD/lb. Butter (lb): 4.20 USD/lb. "
            "Payment terms: Net-45. Early payment discount: 2% if paid within 10 days. "
            "They requested a 12-month contract with quarterly price reviews. "
            "Delivery: Monday-Friday with 24-hour notice. Cold chain guaranteed."
        ),
        "supplier_name": "Globex Dairy Inc.",
        "category": "Dairy",
        "bid_date": "2024-05-02",
        "document_type": "Supplier Quote",
        "status": "Active",
        "delivery_terms": "Mon-Fri (24-hour notice)",
        "price_range": "$1.65-$4.20 per unit"
    },
    
    "bid-summary-2024": {
        "title": "Riverside USD RFP Summary - BID-2024-089",
        "content": (
            "Bid BID-2024-089 for Riverside USD School District: Comprehensive food service RFP. "
            "Eight suppliers solicited across five categories: Produce, Dairy, Proteins, "
            "Beverages, and Pantry. Response rate: five responded (62.5%). "
            "Internal due date: 2024-05-20. Customer due date: 2024-05-25. "
            "Bid opening: 2024-06-01 at 2:00 PM. "
            "Status: Active and under review. Budget allocation: $500,000 annually. "
            "Contract period: July 1, 2024 - June 30, 2025 with renewal options. "
            "Evaluation criteria: price (40%), quality (35%), delivery reliability (15%), service (10%)."
        ),
        "supplier_name": "Riverside USD",
        "category": "RFP Summary",
        "bid_date": "2024-05-01",
        "document_type": "RFP",
        "status": "Active",
        "delivery_terms": "Multi-category",
        "price_range": "$500K annual budget"
    },
    
    "sysco-proteins-2024": {
        "title": "Sysco - Proteins & Meats Quote",
        "content": (
            "Sysco Foodservice quoted for protein category serving Riverside USD on 2024-05-03. "
            "Chicken breast (boneless, skinless, lb): 4.50 USD/lb. "
            "Ground beef 80/20 (lb): 5.20 USD/lb. "
            "Ground beef 90/10 (lb): 5.85 USD/lb. "
            "Salmon fillets (wild-caught, lb): 12.80 USD/lb. "
            "Tilapia fillets (lb): 6.40 USD/lb. "
            "Turkey breast (sliced, lb): 4.95 USD/lb. "
            "Volume discount: 5% on orders over 500 lbs. "
            "Payment terms: Net-60 with 1% early payment discount if paid by 10th. "
            "Delivery: Monday-Friday with 48-hour notice preferred. "
            "Cold chain maintained. USDA inspection certified. Halal options available."
        ),
        "supplier_name": "Sysco Foodservice",
        "category": "Proteins",
        "bid_date": "2024-05-03",
        "document_type": "Supplier Quote",
        "status": "Active",
        "delivery_terms": "Mon-Fri (48-hour notice)",
        "price_range": "$4.50-$12.80 per lb"
    },
    
    "us-foods-beverages-2024": {
        "title": "US Foods - Beverages & Juices Quote",
        "content": (
            "US Foods provided comprehensive beverages quote for Riverside USD on 2024-05-04. "
            "Orange juice concentrate (gallon): 2.30 USD/gallon. "
            "Apple juice (gallon): 1.95 USD/gallon. "
            "Cranberry juice (gallon): 2.75 USD/gallon. "
            "Milk 2% (gallon): 3.15 USD/gallon. "
            "Chocolate milk (gallon): 3.45 USD/gallon. "
            "Bottled water (case/24 bottles): 4.20 USD/case. "
            "Coffee (ground, 2lb bag): 8.50 USD/bag. "
            "Tea assortment (box/50 bags): 6.75 USD/box. "
            "Minimum order: $250. Free delivery for orders over $500. "
            "Payment: Net-45 with 2% early payment discount (10 days). "
            "Special: 10% discount on annual contracts signed by June 30, 2024."
        ),
        "supplier_name": "US Foods Inc.",
        "category": "Beverages",
        "bid_date": "2024-05-04",
        "document_type": "Supplier Quote",
        "status": "Active",
        "delivery_terms": "Flexible (free delivery >$500)",
        "price_range": "$1.95-$8.50 per unit"
    },
    
    "performance-review-2024": {
        "title": "Q1 2024 Supplier Performance Review",
        "content": (
            "Q1 2024 Performance Review Summary: Acme Foods maintained 98% on-time delivery rate, "
            "with quality score of 9.2/10. Customer satisfaction: 4.8/5 stars. "
            "Globex Dairy improved quality from 94% to 97%, on-time delivery 95%. "
            "Excellent communication and responsive to special requests. "
            "Sysco maintained competitive pricing with 96% delivery reliability. "
            "Quality score: 9.0/10. Large variety of product specifications available. "
            "US Foods expanded delivery zones to cover all district schools. "
            "On-time delivery: 92%, quality: 8.8/10. Good value for budget-conscious procurement. "
            "Overall district satisfaction: 8.5/10 across all categories. "
            "Recommendations: Continue contracts with Acme and Globex; negotiate volume discounts "
            "with Sysco; evaluate US Foods for secondary supplier relationship."
        ),
        "supplier_name": "Riverside USD",
        "category": "Performance Review",
        "bid_date": "2024-04-01",
        "document_type": "Internal Report",
        "status": "Active",
        "delivery_terms": "N/A",
        "price_range": "N/A"
    }
})

Wrote 6 document(s) into rag_source_documents.


In [ ]:
# # ---------------------------------------------------------------------------
# # 7a. Managed source-documents table: write your text into SQL, then read it back.
# # ---------------------------------------------------------------------------
# from psycopg2.extras import execute_values

# SOURCE_TABLE = "rag_source_documents"

# def ensure_source_table():
#     """One-time setup: create the table that holds your raw document text. This is
#     separate from the per-combo rag_bench_* tables Step 8 creates, which hold chunks
#     + vectors, not the original document."""
#     with get_conn() as conn, conn.cursor() as cur:
#         cur.execute(f"""
#             CREATE TABLE IF NOT EXISTS {SOURCE_TABLE} (
#                 doc_id text PRIMARY KEY,
#                 title text,
#                 content text NOT NULL,
#                 updated_at timestamptz NOT NULL DEFAULT now()
#             )
#         """)
#         conn.commit()

# def write_source_docs(doc_map: dict[str, str]):
#     """Insert or update documents. doc_map is {doc_id: full_text}. Safe to call
#     repeatedly - it upserts by doc_id, so re-running with the same id updates the text
#     instead of duplicating it, and new ids just get added."""
#     ensure_source_table()
#     rows = [(doc_id, doc_id, text) for doc_id, text in doc_map.items()]
#     with get_conn() as conn, conn.cursor() as cur:
#         execute_values(
#             cur,
#             f"""INSERT INTO {SOURCE_TABLE} (doc_id, title, content) VALUES %s
#                 ON CONFLICT (doc_id) DO UPDATE
#                 SET content = EXCLUDED.content, updated_at = now()""",
#             rows,
#         )
#         conn.commit()
#     print(f"Wrote {len(rows)} document(s) into {SOURCE_TABLE}.")

# def load_source_docs() -> dict[str, str]:
#     """Read every document back out of the managed table, as {doc_id: text}."""
#     with get_conn() as conn, conn.cursor() as cur:
#         cur.execute(f"SELECT doc_id, content FROM {SOURCE_TABLE} ORDER BY doc_id")
#         return {doc_id: content for doc_id, content in cur.fetchall()}

# # Seed the table with the sample documents (safe to re-run — it only upserts, never
# # wipes existing rows). Replace this dict with your own RFP/bid text, or call
# # write_source_docs({...}) again later with more/updated documents.
# write_source_docs({
#     "acme-produce-2024": (
#         "Acme Foods bid on the produce category for Riverside USD on 2024-05-01. "
#         "Romaine lettuce: 3.20 USD/case. Spinach: 4.10 USD/case. Carrots: 2.05 USD/case. "
#         "Payment terms: Net-30. Delivery: twice weekly."
#     ),
#     "globex-dairy-2024": (
#         "Globex Dairy responded to the dairy RFP for Riverside USD. "
#         "Whole milk: 1.85 USD/gallon. Cheddar block: 3.40 USD/lb. Terms: Net-45. "
#         "They offered a 2% early-payment discount and requested a 12-month contract."
#     ),
#     "bid-summary-2024": (
#         "Bid BID-2024-089 for Riverside USD: eight suppliers solicited, five responded "
#         "(62.5% response rate). Internal due 2024-05-20, customer due 2024-05-25. Status: Active."
#     ),
# })

In [ ]:
# ---------------------------------------------------------------------------
# 7b. Or: read documents straight out of a table you already have.
# ---------------------------------------------------------------------------
EXISTING_DOCS_TABLE = "your_existing_table"   # CHANGE - e.g. "documents" or "bids"
EXISTING_ID_COL     = "id"                    # CHANGE - column with a unique id per document
EXISTING_TEXT_COL   = "content"               # CHANGE - column with the document's full text
EXISTING_WHERE      = None                    # optional SQL filter, e.g. "category = 'produce'"

def load_docs_from_existing_table() -> dict[str, str]:
    """Read {id: text} straight from a table that already exists in this database.
    Only used when DOCS_SOURCE = "existing" in 7c below. Rows with empty/NULL text are
    skipped since there's nothing to chunk/embed."""
    where_sql = f"WHERE {EXISTING_WHERE}" if EXISTING_WHERE else ""
    query = f"SELECT {EXISTING_ID_COL}, {EXISTING_TEXT_COL} FROM {EXISTING_DOCS_TABLE} {where_sql}"
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(query)
        return {str(doc_id): text for doc_id, text in cur.fetchall() if text}

print("load_docs_from_existing_table() ready (only used if DOCS_SOURCE = 'existing').")

In [8]:
# ---------------------------------------------------------------------------
# 7c. Pick which source feeds the benchmark, and set your evaluation questions.
# ---------------------------------------------------------------------------
# "managed"  -> read back what write_source_docs() wrote in 7a (recommended default)
# "existing" -> read from EXISTING_DOCS_TABLE/EXISTING_ID_COL/EXISTING_TEXT_COL in 7b
DOCS_SOURCE = "managed"

if DOCS_SOURCE == "managed":
    docs = load_source_docs()
elif DOCS_SOURCE == "existing":
    docs = load_docs_from_existing_table()
else:
    raise ValueError(f"Unknown DOCS_SOURCE: {DOCS_SOURCE}")

# gold: question -> the doc_id that should be retrieved for it. Used in Step 10 to score
# each (model, chunker, indexer) combo objectively instead of eyeballing results.
# IMPORTANT: these doc_id values must match keys in `docs` above - update them once you
# switch DOCS_SOURCE to your own real documents/questions.
gold = {
    "What did Acme quote for romaine?": "acme-produce-2024",
    "Which supplier offered an early payment discount?": "globex-dairy-2024",
    "What were Globex's payment terms?": "globex-dairy-2024",
    "What was the supplier response rate?": "bid-summary-2024",
    "When is the customer due date?": "bid-summary-2024",
}

# Sanity check: catch a mismatch between gold's doc_ids and what docs actually contains
# (e.g. you switched DOCS_SOURCE but forgot to update gold) before it silently tanks recall.
missing_gold = [doc_id for doc_id in gold.values() if doc_id not in docs]
if missing_gold:
    print(f"⚠ {len(missing_gold)} gold doc_id(s) not found in docs: {missing_gold}")

print(f"{len(docs)} documents loaded (source={DOCS_SOURCE}), {len(gold)} evaluation questions.")

6 documents loaded (source=managed), 5 evaluation questions.


## Step 8 — Ingest: chunk + embed + store (one table per combo)

For each (embedding model x chunking method x index type) we build a separate table so results are comparable. The vector dimension is **auto-detected** from the model, so mixing 768-dim and 3072-dim models just works.

In [9]:
from psycopg2.extras import execute_values

def table_name(model, chunker, indexer):
    """Build a safe, unique table name per (model, chunker, indexer) combo so results
    from different combos never overwrite each other."""
    slug = re.sub(r"[^a-z0-9]+", "_", f"{model}_{chunker}_{indexer}".lower()).strip("_")
    return f"rag_bench_{slug}"[:60]     # stay under Postgres's 63-char identifier limit

def create_index(cur, table, indexer, n_rows):
    """Create the chosen storage/search index on `table`, right after its rows are
    inserted. This is the 'storing' dimension we benchmark in Step 10."""
    if indexer == "hnsw":
        # Hierarchical Navigable Small World graph: approximate nearest-neighbour index,
        # generally the best default for production-sized data (fast search, no training step).
        cur.execute(f"CREATE INDEX ON {table} USING hnsw (embedding vector_cosine_ops)")
    elif indexer == "ivfflat":
        # Inverted File index: approximate, cheaper to build than hnsw but needs a
        # "lists" parameter and a data-dependent training pass. Rule of thumb is roughly
        # sqrt(n_rows); we floor it at 1 because our demo tables only have a few rows.
        lists = max(1, n_rows // 10)
        cur.execute(
            f"CREATE INDEX ON {table} USING ivfflat (embedding vector_cosine_ops) "
            f"WITH (lists = {lists})"
        )
        cur.execute(f"ANALYZE {table}")   # refresh planner statistics so Postgres will use the new index
    elif indexer == "none":
        pass  # No index created -> queries fall back to an exact, row-by-row scan.
              # Slower as the table grows, but always exact, and (unlike hnsw/ivfflat)
              # has no 2000-dimension ceiling - see the gemini-embedding-001 note in Step 10.
    else:
        raise ValueError(f"Unknown indexer: {indexer}")

def ingest(model, chunker, indexer):
    """Chunk every doc with `chunker`, embed the chunks with `model`, store them in a
    fresh table built with `indexer`. Returns (table_name, vector_dim, n_chunks)."""
    fn = CHUNKER_FUNCS[chunker]
    rows = []                       # (doc_id, chunk_index, content) for every chunk of every doc
    for doc_id, text in docs.items():
        for idx, piece in enumerate(fn(text)):
            rows.append((doc_id, idx, piece))

    # Embed every chunk. task_type=RETRIEVAL_DOCUMENT tells the model these are the
    # "things that can be found", as opposed to the question doing the finding.
    vectors = embed_texts(model, [r[2] for r in rows], "RETRIEVAL_DOCUMENT")
    dim = len(vectors[0])
    table = table_name(model, chunker, indexer)

    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {table}")
        # embedding vector(dim): the column is sized exactly to this model's output length.
        cur.execute(f"""
            CREATE TABLE {table} (
                id serial PRIMARY KEY,
                doc_id text, chunk_index int, content text, embedding vector({dim})
            )
        """)
        # Bulk-insert every chunk row in a single round trip. Each Python list of floats
        # is passed as its string form and cast to Postgres's `vector` type via ::vector.
        execute_values(
            cur,
            f"INSERT INTO {table} (doc_id, chunk_index, content, embedding) VALUES %s",
            [(rows[i][0], rows[i][1], rows[i][2], str(vectors[i])) for i in range(len(rows))],
            template="(%s, %s, %s, %s::vector)",
        )
        create_index(cur, table, indexer, len(rows))
        conn.commit()
    return table, dim, len(rows)

print("ingest() ready.")

ingest() ready.


## Step 9 — Vector search

`<=>` is pgvector's cosine distance (0 = identical meaning). We embed the question with the **same model** and return the closest pieces. This works the same regardless of which index type (or no index) the table was built with — the index only changes *how fast* this runs, not the query itself.

In [10]:
import time   # used in Step 10 to time how long each search takes

def search(table, model, question, top_k=TOP_K):
    """Embed `question` with the same model used to embed the documents, then ask
    Postgres for the `top_k` closest chunks by cosine distance."""
    # RETRIEVAL_QUERY tunes the embedding for "this is a question", matching the
    # RETRIEVAL_DOCUMENT task_type used for the stored chunks in Step 8.
    qv = embed_texts(model, [question], "RETRIEVAL_QUERY")[0]
    with get_conn() as conn, conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        # ORDER BY distance ASC + LIMIT top_k = "give me the k closest chunks".
        cur.execute(f"""
            SELECT doc_id, content, embedding <=> %s::vector AS distance
            FROM {table} ORDER BY distance LIMIT %s
        """, (str(qv), top_k))
        return [dict(r) for r in cur.fetchall()]

print("search() ready.")

search() ready.


## Step 10 — Benchmark every combo → leaderboard

For each combo (model x chunking x index type) we ingest, then score the gold questions:
- **recall@k** — how often the correct document is in the top-k.
- **MRR** — how *high* it ranks (1.0 = always first).
- **latency (ms)** — how long the search itself took on average; this is where the index type (`hnsw` / `ivfflat` / `none`) shows its effect. On this tiny demo dataset all index types will likely score the same recall/MRR — the gap only shows up at real-world data volumes, but the latency numbers still hint at the trend.

Combos whose model/index isn't compatible are skipped and reported (they won't stop the run) — e.g. `gemini-embedding-001` (3072 dims) is skipped for `hnsw`/`ivfflat`, which cap at 2000 dims, but works fine under `none`.

In [11]:
def score(table, model):
    """Run every gold question against `table` and return (recall@k, MRR, avg latency ms)."""
    recall_hits, rr_sum, latency_sum = 0, 0.0, 0.0
    for query, expected in gold.items():
        start = time.perf_counter()
        ranked = [h["doc_id"] for h in search(table, model, query, TOP_K)]
        latency_sum += (time.perf_counter() - start) * 1000     # seconds -> milliseconds
        if expected in ranked:
            recall_hits += 1
            rr_sum += 1.0 / (ranked.index(expected) + 1)        # rank 1 -> 1.0, rank 2 -> 0.5, ...
    n = len(gold)
    return recall_hits / n, rr_sum / n, latency_sum / n

results = []
# Try every (embedding model) x (chunking method) x (index type) combination -
# this is the full 3-D benchmark: model x chunking x storing.
for model in EMBEDDERS:
    for chunker in CHUNKERS:
        for indexer in INDEXERS:
            label = f"{model} | {chunker} | {indexer}"
            try:
                table, dim, n = ingest(model, chunker, indexer)
                recall, mrr, latency_ms = score(table, model)
                results.append({"label": label, "recall": recall, "mrr": mrr,
                                "latency_ms": latency_ms, "table": table, "dim": dim, "ok": True})
                print(f"✓ {label:<58} dim={dim:<5} recall@{TOP_K}={recall:.2f} mrr={mrr:.2f} "
                      f"latency={latency_ms:.1f}ms")
            except Exception as e:
                # A combo can legitimately fail (e.g. a model not enabled in your project/region,
                # or an index type that can't handle this many dimensions) - log and keep going
                # instead of stopping the whole benchmark.
                results.append({"label": label, "ok": False, "error": f"{type(e).__name__}: {e}"})
                print(f"✗ {label:<58} skipped ({str(e)[:60]})")

✓ text-embedding-004 | fixed_400 | hnsw                      dim=768   recall@3=0.80 mrr=0.53 latency=3266.9ms
✓ text-embedding-004 | fixed_400 | ivfflat                   dim=768   recall@3=0.80 mrr=0.53 latency=3244.8ms
✓ text-embedding-004 | fixed_400 | none                      dim=768   recall@3=0.80 mrr=0.53 latency=3225.0ms
✓ text-embedding-004 | sentence | hnsw                       dim=768   recall@3=0.80 mrr=0.60 latency=3218.9ms
✓ text-embedding-004 | sentence | ivfflat                    dim=768   recall@3=0.80 mrr=0.60 latency=3204.8ms
✓ text-embedding-004 | sentence | none                       dim=768   recall@3=0.80 mrr=0.60 latency=3452.7ms
✓ text-embedding-004 | token_256 | hnsw                      dim=768   recall@3=0.80 mrr=0.70 latency=3200.9ms
✓ text-embedding-004 | token_256 | ivfflat                   dim=768   recall@3=0.80 mrr=0.70 latency=3208.9ms
✓ text-embedding-004 | token_256 | none                      dim=768   recall@3=0.80 mrr=0.70 latency=3300.9ms
✓

In [12]:
# Rank combos by quality first (MRR, then recall@k); latency is shown for visibility and
# only used as a tiebreaker, since a faster-but-wrong combo isn't actually "better".
ok = sorted([r for r in results if r["ok"]],
            key=lambda r: (r["mrr"], r["recall"], -r["latency_ms"]), reverse=True)

print(f"{'rank':<5}{'combo':<58}{'recall@'+str(TOP_K):<12}{'mrr':<8}{'latency(ms)':<12}")
print("-" * 95)
for i, r in enumerate(ok, 1):
    print(f"{i:<5}{r['label']:<58}{r['recall']:<12.2f}{r['mrr']:<8.2f}{r['latency_ms']:<12.1f}")

if ok:
    best = ok[0]
    print(f"\n⭐ Best for your data: {best['label']}  (table: {best['table']})")
    # "model | chunker | indexer" -> pull the model name back out for Step 11.
    BEST_MODEL, _BEST_CHUNKER, _BEST_INDEXER = [p.strip() for p in best["label"].split("|")]
    BEST_TABLE = best["table"]

rank combo                                                     recall@3    mrr     latency(ms) 
-----------------------------------------------------------------------------------------------
1    gemini-embedding-001 | token_256 | none                   1.00        0.90    3660.8      
2    gemini-embedding-001 | sentence | none                    1.00        0.77    3542.8      
3    gemini-embedding-001 | fixed_400 | none                   1.00        0.73    3562.6      
4    text-embedding-005 | fixed_400 | none                     1.00        0.70    3236.4      
5    text-embedding-005 | fixed_400 | hnsw                     1.00        0.70    3246.7      
6    text-embedding-005 | fixed_400 | ivfflat                  1.00        0.70    3256.9      
7    text-embedding-004 | token_256 | hnsw                     0.80        0.70    3200.9      
8    text-embedding-004 | token_256 | ivfflat                  0.80        0.70    3208.9      
9    text-embedding-004 | token_256 | no

## Step 11 — Answer questions with Gemini (using the winning combo)

`ask()` retrieves from the best table and lets Gemini answer from those pieces only.

In [13]:
def ask(question, top_k=TOP_K):
    """Retrieve the top_k chunks from the winning table, then have Gemini answer
    strictly from that context (reduces made-up answers / hallucination)."""
    hits = search(BEST_TABLE, BEST_MODEL, question, top_k)
    context = "\n\n---\n\n".join(h["content"] for h in hits)
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the context doesn't contain the answer, say you don't know.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}"
    )
    resp = genai_client.models.generate_content(model=CHAT_MODEL, contents=prompt)
    return resp.text

print(ask("What was the Customer Satisfaction level for supplier performance review?"))

Overall district satisfaction was 8.5/10 across all categories.


## Step 12 — Notes & cleanup

**Reading the leaderboard**
- Pick the top combo; that's your best `model + chunking + storage/index` for this data.
- With only 5 gold questions the scores are coarse — add 20–50 real questions for a trustworthy result.
- On recall/MRR ties, prefer the smaller/cheaper model (768-dim is cheaper to store/search than 3072-dim) and the lower-latency index.

**How to think about the storing/index dimension**
- `hnsw` — best general-purpose default; approximate but very accurate in practice; no training step.
- `ivfflat` — cheaper to build than hnsw, but quality depends on tuning `lists` and having enough rows to "train" on; on tiny tables like this demo it behaves close to brute force.
- `none` — exact, always correct, no dimension limit, but a full table scan on every query — fine for small tables, gets slow as data grows into the tens/hundreds of thousands of rows. This is also the only option that supports `gemini-embedding-001`'s 3072-dim vectors, since pgvector's hnsw/ivfflat index types cap at 2000 dimensions.

**Going further (better results)**
- Add **hybrid search** (vector + keyword) and a **reranker** — already implemented in `services/rag/` (`docs/rag_pgvector_guide.md`).
- Keep the winning table for production instead of dropping it.

The cleanup below drops the temporary `rag_bench_*` tables. Skip it if you want to keep the winner.

In [ ]:
with get_conn() as conn, conn.cursor() as cur:
    for r in results:
        if r.get("table"):
            cur.execute(f"DROP TABLE IF EXISTS {r['table']}")
    conn.commit()
print("Temporary benchmark tables dropped.")

## Troubleshooting

| Symptom | Fix |
|---|---|
| `could not connect to server` | DB_CONFIG wrong, or Cloud SQL Auth Proxy not running (Step 2/3) |
| `403 PERMISSION_DENIED` from Vertex | `gcloud auth application-default login` + enable the Vertex AI API |
| `type "vector" does not exist` | Re-run Step 4 (enable pgvector) against the right database |
| A model is skipped in Step 10 | That model isn't enabled in your project/region — remove it from `EMBEDDERS` |
| `gemini-embedding-001` skipped under `hnsw`/`ivfflat` | Expected — 3072 dims exceeds pgvector's 2000-dim index limit; it still works under `none` (no index) |
| Search slow on a big table | Use `hnsw` (Step 8/10); for very large tables also tune `ivfflat`'s `lists` value |
| Good chunks, weak answers | raise `TOP_K`, or improve the prompt in Step 11 |